# Spatial Filtering
Middlebury M

> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-04.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** run the setup cell below first. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

In [ ]:
# --- Setup: run this cell first.  It does nothing outside Google Colab. ---
import sys, os, importlib.util, urllib.request
IN_COLAB = "google.colab" in sys.modules
DATA_FILES = ['figures/middleburyM400x400.png']  # fetched so the notebook's relative paths work in Colab
if IN_COLAB:
    for _f in DATA_FILES:
        _dest = os.path.join("..", _f)
        if not os.path.exists(_dest):
            os.makedirs(os.path.dirname(_dest), exist_ok=True)
            urllib.request.urlretrieve("https://raw.githubusercontent.com/durstme/textbook-biomedicalimaging-notebooks/main/" + _f, _dest)

## Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import ipywidgets as widgets
from IPython.display import display

In [2]:
%matplotlib inline
# Define global "bold" style parameters
# This restores the thicker lines and fonts seen in older versions
plot_params = {
    # --- Fonts ---
    'font.size': 18,
    'font.family': 'sans-serif',
    
    # --- Axes (The Box) ---
    'axes.linewidth': 1.0,       # Thicker box around the plot
    
    # --- Lines (The Signal) ---
    'lines.linewidth': 2.5,      # Thicker default plot lines
}

# Apply these settings globally
plt.rcParams.update(plot_params)

## Define parameters

In [3]:

# --- 1. LOAD IMAGE HELPER ---
def load_and_prep_image(filepath):
    try:
        img = mpimg.imread(filepath)
    except FileNotFoundError:
        print(f"File {filepath} not found. Using random noise.")
        img = np.random.rand(400, 400)

    # Use Green channel if RGB
    if img.ndim == 3:
        data = img[:, :, 1] 
    else:
        data = img

    # Normalize
    if data.max() > 1:
        data = data / 255.0
        
    # Pre-compute FFT
    fft_raw = np.fft.fft2(data)
    fft_shifted = np.fft.fftshift(fft_raw)
    
    return data, fft_shifted

# --- 2. MASK LOGIC (Same as before) ---
def create_mask(shape, mask_type, width):
    rows, cols = shape
    cx, cy = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    mask = np.ones(shape)
    r = max(1, width)

    if mask_type == 0:   # Mask 1: Black Circle (High Pass)
        mask[(x - cx)**2 + (y - cy)**2 <= r**2] = 0
    elif mask_type == 1: # Mask 2: White Circle (Low Pass)
        mask = np.zeros(shape)
        mask[(x - cx)**2 + (y - cy)**2 <= r**2] = 1
    elif mask_type == 2: # Mask 3: Horiz Split
        mask[:cx, :] = 0; mask[cx:, :] = 1
    elif mask_type == 3: # Mask 4: Vert Split
        mask[:, :cy] = 0; mask[:, cy:] = 1
    elif mask_type == 4: # Mask 5: Horiz White Bar
        mask = np.zeros(shape)
        mask[cx-r:cx+r, :] = 1
    elif mask_type == 5: # Mask 6: Vert White Bar
        mask = np.zeros(shape)
        mask[:, cy-r:cy+r] = 1
    elif mask_type == 6: # Mask 7: Horiz Black Bar
        mask[cx-r:cx+r, :] = 0
    elif mask_type == 7: # Mask 8: Vert Black Bar
        mask[:, cy-r:cy+r] = 0
        
    return mask

# --- 3. LOAD DATA ONCE ---
filename1 = "../figures/middleburyM400x400.png" 
original_image1, fft_shifted1 = load_and_prep_image(filename1)
fft_mag_log1 = np.log(1 + np.abs(fft_shifted1))



## Plot

In [4]:
%matplotlib inline
# --- 4. INTERACTIVE UPDATE FUNCTION ---
def update_view(mask_index, width):
    # 1. Generate Mask
    mask = create_mask(original_image1.shape, mask_index, width)
    
    # 2. Apply FFT Math
    masked_fft = fft_shifted1 * mask
    inv_img = np.abs(np.fft.ifft2(np.fft.ifftshift(masked_fft)))
    
    # 3. Create Plot (2x2 Grid)
    fig1, axes = plt.subplots(2, 2, figsize=(10, 10))
    
    # Flatten the 2D array of axes so we can index them simply as 0,1,2,3
    ax = axes.flatten()
    
    # Plot Original (Top-Left)
    ax[0].imshow(original_image1, cmap='gray')
    ax[0].set_title("Original Image")
    ax[0].axis('off')
    
    # Plot FFT (Top-Right)
    ax[1].imshow(fft_mag_log1, cmap='gray')
    ax[1].set_title("Fourier Transform (Log Mag)")
    ax[1].axis('off')
    
    # Plot Mask (Bottom-Right)
    ax[3].imshow(mask, cmap='gray', vmin=0, vmax=1)
    ax[3].set_title(f"Mask (Width={width})")
    ax[3].axis('off')
    
    # Plot Result (Bottom-Left)
    ax[2].imshow(inv_img, cmap='gray')
    ax[2].set_title("Filtered Result")
    ax[2].axis('off')
    
    plt.tight_layout()
    # plt.savefig("../figures/ch05/ch05-04-FTspatialfilter-still.jpg", dpi=300, bbox_inches='tight')
    plt.show()

# --- 5. CREATE WIDGETS ---
mask_dropdown = widgets.Dropdown(
    options=[
        ('Mask 1: High Pass (Black Circle)', 0),
        ('Mask 2: Low Pass (White Circle)', 1),
        ('Mask 3: Horizontal Edge', 2),
        ('Mask 4: Vertical Edge', 3),
        ('Mask 5: Horizontal Pass', 4),
        ('Mask 6: Vertical Pass', 5),
        ('Mask 7: Horizontal Stop', 6),
        ('Mask 8: Vertical Stop', 7),
    ],
    value=0,
    description='Mask Shape:',
)

width_slider = widgets.IntSlider(
    value=30,
    min=1,
    max=100,
    step=1,
    description='Mask Width:',
    continuous_update=False 
)

widgets.interact(update_view, mask_index=mask_dropdown, width=width_slider);

### Display animation with javascript from standalone html
[Link to animation](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/animations/ch05/ch05-04_animation.html)

*Animation.* The pre-rendered animation for this section is shown on the [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-04.html) (direct link: [ch05-04_animation.html](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/animations/ch05/ch05-04_animation.html)). It is not embedded here because Colab cannot load files from the textbook site; running the code above produces the live version.